# SwarKavach: run the whole thing

One notebook, start to finish. It fetches the real corpora, builds the Hindi
anti-spoofing set, generates the Hinglish call corpus with neural voices,
trains every model, evaluates, and hands you a zip to drop back into the repo.

**Runtime**: Runtime, Change runtime type, **T4 GPU**. It works on CPU too,
just slower, and the notebook says so where it matters.

**Time**: about 45 to 70 minutes. The slow parts are the dataset download and
the speech synthesis, neither of which needs the GPU.

**Why here and not on a laptop**: the corpus build holds a lot of audio in
memory, and the anti-spoofing branch is happier with room to work. Colab gives
you roughly 13 GB of RAM against the 8 GB on the machine this was written on,
where the same run kept getting killed.

**What you get back**: `data/results/*.json` for the console, and the trained
models. Unzip both into your local checkout and run `swarkavach serve`.


## 0. Clone and install


### Your GitHub token

The repo can be private. Put a Personal Access Token in Colab's **Secrets**
panel (the key icon in the left sidebar) under the name `GH_PAT`, and switch
on "Notebook access". A fine-grained token needs **Contents: read and write**
on this one repository, nothing else.

Secrets are the right place for it. Pasting a token into a cell puts it in the
notebook file, which then travels with any copy you share. Nothing below ever
prints it, and the token is stripped from the git remote right after cloning
so it is not left sitting in `.git/config`.

One consequence worth knowing: `!git pull` on its own will fail with
`could not read Username for 'https://github.com'`, because there is no
credential left in the clone for it to use. To update, re-run the cell below
rather than pulling by hand. It re-authenticates for one fetch and cleans up
after itself.


In [ ]:
# CHANGE THIS to your own repo.
REPO_SLUG = "BhavyeNijhawan/svarkavach"     # user/repo
BRANCH    = "main"

import os, subprocess, shutil, getpass
from pathlib import Path

# Deliberately NOT "swarkavach". Colab keeps /content on sys.path, so a
# directory there whose name matches the package shadows the real install
# as an empty namespace package, and you get "cannot import name config
# from swarkavach (unknown location)" much later on. A different directory
# name avoids the collision entirely.
CLONE_DIR = "/content/svk_repo" if Path("/content").is_dir() else "svk_repo"

TOKEN = ""
try:
    from google.colab import userdata
    TOKEN = userdata.get("GH_PAT") or ""
except Exception:
    pass
if not TOKEN:
    print("No GH_PAT secret found. Leave blank if the repo is public.")
    TOKEN = getpass.getpass("GitHub token (hidden, optional): ").strip()

# never build a URL with the token into a printable variable
def _auth_url():
    return (f"https://{TOKEN}@github.com/{REPO_SLUG}.git" if TOKEN
            else f"https://github.com/{REPO_SLUG}.git")

CLEAN_URL = f"https://github.com/{REPO_SLUG}.git"
print("repo:", CLEAN_URL, "| token:", "yes" if TOKEN else "no")


In [ ]:
# Re-run THIS cell to update the clone. A bare `!git pull` in a cell of your
# own will fail with "could not read Username for https://github.com", because
# the token is deliberately stripped from .git/config at the bottom of this
# cell. That is the point: a private repo's credential should not sit in a
# file inside a runtime other people can be handed. This cell puts it back for
# the length of one fetch and removes it again.
if Path(f"{CLONE_DIR}/.git").exists():
    print("already cloned, updating")
    subprocess.run(["git", "-C", CLONE_DIR, "remote", "set-url", "origin", _auth_url()],
                   check=False, capture_output=True)

    def _head():
        return subprocess.run(["git", "-C", CLONE_DIR, "rev-parse", "--short", "HEAD"],
                              capture_output=True, text=True).stdout.strip()

    before = _head()
    r = subprocess.run(["git", "-C", CLONE_DIR, "pull", "--ff-only", "origin", BRANCH],
                       capture_output=True, text=True)
    if r.returncode != 0:
        msg = (r.stderr or "") + (r.stdout or "")
        if TOKEN:
            msg = msg.replace(TOKEN, "***")
        subprocess.run(["git", "-C", CLONE_DIR, "remote", "set-url", "origin", CLEAN_URL],
                       check=False, capture_output=True)
        print(f"pull failed, still on {before}")
        print(msg[-700:])
        print()
        print("Most often a run wrote inside the clone, or Colab autosaved a")
        print("notebook. Nothing here is worth keeping, so take the remote copy:")
        print(f"  !git -C {CLONE_DIR} fetch origin {BRANCH} && \\")
        print(f"   git -C {CLONE_DIR} reset --hard origin/{BRANCH}")
        print("then re-run this cell.")
        # Loud on purpose. Silently continuing on stale code means the next
        # hour of synthesis and training tests the wrong commit.
        raise SystemExit("not on the latest code, stopping")
    after = _head()
    print(f"updated {before} -> {after}" if before != after else f"already current at {after}")
else:
    shutil.rmtree(CLONE_DIR, ignore_errors=True)
    r = subprocess.run(["git", "clone", "--branch", BRANCH, _auth_url(), CLONE_DIR],
                       capture_output=True, text=True)
    if r.returncode != 0:
        # scrub the token out of anything we show
        msg = (r.stderr or "").replace(TOKEN, "***") if TOKEN else (r.stderr or "")
        raise SystemExit("clone failed: " + msg[-800:])
    print("cloned")

# strip the token back out of the stored remote
subprocess.run(["git", "-C", CLONE_DIR, "remote", "set-url", "origin", CLEAN_URL],
               check=False, capture_output=True)

os.chdir(CLONE_DIR)
print("cwd:", os.getcwd())
_r = subprocess.run(["git", "remote", "-v"], capture_output=True, text=True).stdout.splitlines()
print("remote:", _r[0] if _r else "none")
_log = subprocess.run(["git", "log", "-1", "--format=%h  %s"],
                      capture_output=True, text=True).stdout.strip()
print("at:", _log)


In [ ]:
# Install. Colab already has torch, numpy and friends, so this is mostly small
# packages plus an editable install of the project itself.
!pip install -q -r requirements.txt 2>&1 | tail -3
!pip install -q edge-tts indic-transliteration soundfile 2>&1 | tail -3
!pip install -q -e . 2>&1 | tail -3

# Verify rather than assume. A quiet pip failure here shows up much later as a
# confusing ImportError, so check it now and say exactly what went wrong.
import importlib, sys
from pathlib import Path

src = Path.cwd() / "src"
if src.is_dir() and str(src) not in sys.path:
    sys.path.insert(0, str(src))      # belt and braces if the editable install failed

for m in list(sys.modules):
    if m == "swarkavach" or m.startswith("swarkavach."):
        del sys.modules[m]

try:
    import swarkavach
    from swarkavach import config
    print("swarkavach imports from:", swarkavach.__file__)
    print("version:", swarkavach.__version__)
except ImportError as exc:
    print("swarkavach did not import:", exc)
    print("cwd:", Path.cwd(), "| src exists:", src.is_dir())
    print("Re-run the clone cell, then this one.")
    raise


In [ ]:
# Environment check. Nothing below needs a GPU except the optional RawNet and
# MuRIL steps, but it is worth knowing what you have.
import shutil, sys, torch
from swarkavach import config

print("python      :", sys.version.split()[0])
print("torch       :", torch.__version__, "| cuda:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu         :", torch.cuda.get_device_name(0))
try:
    import psutil
    print("ram         : %.1f GB" % (psutil.virtual_memory().total / 1e9))
except Exception:
    pass
print("disk free   : %.1f GB" % (shutil.disk_usage('.').free / 1e9))
print("ffmpeg      :", "yes" if config.find_ffmpeg() else "no (G.711 still exact, in numpy)")


In [ ]:
# The test suite, before anything expensive runs. It builds its own tiny
# corpus in a temp directory, so it neither reads nor damages data/.
# A failure here means the clone is broken and the next hour would be
# wasted, so it is worth the two minutes.
!python -m pytest tests -q -x 2>&1 | tail -25


### Drive checkpoints

Colab throws the local disk away when the session ends, and free sessions end
often. Every expensive artifact is copied to your Drive as it is produced and
pulled back at the start of the next run, so a disconnect costs you the stage
you were in the middle of and nothing before it.

The one that actually matters is the speech cache. Building it is thousands of
rate-limited synthesis requests and takes about an hour. Building it twice is
an hour you do not get back.

You will be asked to authorise Drive once. Everything lands in
`MyDrive/swarkavach_checkpoints/`, and you can delete that folder any time you
want a genuinely clean run.


In [ ]:
# Set this to redo a stage even though Drive already has its output.
# Valid names: "download", "pairs", "corpus", "train", "evaluate".
FORCE_REBUILD = []

from swarkavach import colab

colab.mount()
colab.restore_all()

DONE = [s for s in colab.what_can_be_skipped() if s not in FORCE_REBUILD]
print()
print("will be skipped:", ", ".join(DONE) if DONE else "nothing, this is a fresh run")
print()
colab.print_status()


def run(cmd):
    """Run one pipeline step. Stops the notebook if it fails.

    Plain subprocess rather than the ! magic because ! swallows the exit code,
    and a training step that died deserves to stop the run rather than let the
    next cell evaluate models that were never written.
    """
    import subprocess
    print("$", cmd, flush=True)
    if subprocess.run(cmd, shell=True).returncode != 0:
        raise SystemExit(f"step failed: {cmd}")


## 1. Data

Two real corpora, both direct HTTP with no account and no approval form:

- **GramVaani** (OpenSLR 118): real Hindi *telephone* speech, transcribed, with
  gender and accent labels. This is the bonafide side of the voice branch.
- **NCSU WSPR robocalls**: 1,432 genuine illegal robocalls with transcripts,
  public domain because they are FTC evidence exhibits. Only the transcripts
  are used, so the clone is blob-filtered down to the 700 KB metadata file
  rather than the full 1.7 GB. They are the out-of-domain test in section 5:
  real fraud scripts, written by real fraudsters, that nothing here was tuned
  on. Pass `--robocall-audio` if you want the recordings as well.

Then it builds the Hindi anti-spoofing pairs. No public Hindi anti-spoofing set
exists, so each spoof renders the **same transcript** as its bonafide, and both
sides go through the same channel with the noise floor matched. Without that
matching the two classes are separable on silence level alone at AUC 0.97, and
the detector learns "quiet means fake" instead of anything about voice.


In [ ]:
# About 800 MB for the core sets. Add "all" instead of "core" for the extra
# 2 GB GramVaani training archive; you do not need it for this run.
if "download" in DONE and "pairs" in DONE:
    print("corpora and anti-spoofing pairs came back from Drive, nothing to fetch")
else:
    # already have the pairs but not the archives, or the other way round
    n_pairs = 0 if "pairs" in DONE else 600
    run(f"python -m swarkavach.cli fetch-data --which core --pairs {n_pairs}")
    colab.save("raw")
    colab.save("robocall")
    if n_pairs:
        colab.save("pairs")


In [ ]:
# Confirm the anti-spoofing set is usable BEFORE training on it. If the worst
# channel statistic scores much above 0.62, the two classes differ in
# recording conditions and any EER from them would be measuring the microphone
# rather than the speaker.
import json
from swarkavach.datasets import load_antispoof_pairs

m = load_antispoof_pairs()
a = m.get("confound_audit", {})
print("pairs :", m.get("n_pairs"))
print("audit :", a.get("status"), "| worst statistic AUC", a.get("worst_auc"))
print()
print(f"{'statistic':16s} {'bonafide':>10} {'spoof':>10} {'AUC alone':>10}")
for k, v in (a.get("per_statistic") or {}).items():
    print(f"{k:16s} {v['bonafide_mean']:10.2f} {v['spoof_mean']:10.2f} {v['auc_alone']:10.3f}")

if (a.get("worst_auc") or 0) > 0.75:
    print("\nSTOP: this set has a channel leak. Do not trust an EER from it.")


## 2. The Hinglish call corpus

Generated dialogues with gold BIO entities, gold dialogue acts and gold
per-token language tags, all emitted by the template slots rather than
annotated by hand. Rendered with real neural voices so the audio says what the
transcript says.

`--backend edge` is the important flag. The `sim` backend produces
speech-shaped noise with the right acoustic statistics and no actual words,
which is fine for training a detector and useless for a demo.


In [ ]:
# 480 calls, roughly 5,000 turns. The synthesis service rejects concurrent
# bursts, so this runs at low concurrency and takes a while. Every line is
# cached by content hash and the cache goes to Drive, so a second run is close
# to free even if this one dies halfway.
if "corpus" in DONE:
    print("corpus came back from Drive, skipping generation")
else:
    try:
        run("python -m swarkavach.cli gen-corpus --n 480 --audio --backend edge")
    finally:
        # save the cache even on a failure, since that is the hour you would
        # otherwise have to spend again
        colab.save("tts_cache")
    colab.save("corpus")


In [ ]:
# Listen to one. Pick a cloned scam call and check the audio matches the text.
import glob, random
from IPython.display import Audio, display
from swarkavach.schema import Call

paths = sorted(glob.glob("data/corpus/calls/*.json"))
cands = [Call.load(p) for p in paths[:80]]
call = next((c for c in cands if c.label_scam and c.label_voice == "synthetic"), cands[0])

print(f"{call.call_id}  {call.scenario}  {call.cell}\n")
for t in call.turns:
    print(f"  [{t.speaker:6s} {t.act:18s}] {t.text}")
if call.audio_path:
    display(Audio(call.audio_path))


## 3. Train

Order matters here, and the splits are not casual:

- entity tagger, intent model and dialogue-act model fit on **train**
- prosody-intent anchors are recalibrated from the corpus audio
- the anti-spoofing branch fits on the **real paired set**, not the generated
  corpus, because the corpus cannot answer "person or machine" when both of
  its cells are machines
- the fusion layer fits on **dev**, never on train. It stacks on top of the
  branches, so fitting it on rows the branches trained on feeds it in-sample
  predictions that never occur at test time


In [ ]:
if "train" in DONE:
    print("trained models came back from Drive, skipping training")
else:
    run("python -m swarkavach.cli train")
    colab.save("models")


In [ ]:
# Read the anchor warning if there is one. If a component reads higher for
# synthetic speech than for human, it is fighting the prosody-intent feature
# rather than helping it, and needs replacing. That already happened once, to
# a frame-level energy measure.
import json
from pathlib import Path

p = Path("data/models/pim_anchors.json")
if p.exists():
    d = json.loads(p.read_text())
    print("anchors:", json.dumps(d.get("anchors"), indent=2))
    print("\ndirection (True means it points the right way):")
    for k, ok in (d.get("direction_ok") or {}).items():
        med = (d.get("medians") or {}).get(k, {})
        flag = "ok" if ok else "WRONG WAY"
        print(f"  {k:14s} human {med.get('human')}  synthetic {med.get('synthetic')}   {flag}")


## 4. Evaluate

Writes `data/results/*.json`, which is exactly what the local console reads.


In [ ]:
if "evaluate" in DONE:
    print("results came back from Drive, skipping evaluation")
else:
    run('python -m swarkavach.cli evaluate --codecs "clean,g711u,g711a,gsm"')
    colab.save("results")


## 5. Results

The ablation table is the headline. It is reported **per cell** rather than as
one aggregate, because a single number hides the failure that matters: audio
alone should miss human scams, text alone should miss the lexically mild scams
and false-alarm on cloned benign calls, and the full arm should clear both.


In [ ]:
import json
from pathlib import Path

def load(name):
    p = Path("data/results") / name
    return json.loads(p.read_text()) if p.exists() else None

CELL = {"humanxbenign": "Human/benign", "humanxscam": "Human/scam",
        "clonedxbenign": "Cloned/benign", "clonedxscam": "Cloned/scam"}
ARM = {"audio_only": "Audio only", "text_only": "Text only",
       "late_fusion": "Late fusion", "full": "Full (cross-modal)"}

ab = load("ablation_results.json")
if ab:
    cells_ = ab.get("cells", [])
    print("ABLATION, correct decisions per cell, held-out test split\n")
    hdr = f"{'arm':20s}" + "".join(f"{CELL.get(c, c):>16s}" for c in cells_) + f"{'AUC':>8s}{'F1':>8s}"
    print(hdr); print("-" * len(hdr))
    for arm in ab.get("arms", []):
        row = f"{ARM.get(arm, arm):20s}"
        for c in cells_:
            v = ab["detection_rate"].get(arm, {}).get(c)
            row += f"{('n/a' if v is None else format(v, '.0%')):>16s}"
        o = ab.get("overall", {}).get(arm, {})
        row += f"{o.get('auc', float('nan')):>8.3f}{o.get('f1', float('nan')):>8.3f}"
        print(row)


In [ ]:
asr_ = load("antispoof_results.json")
if asr_:
    print("ANTI-SPOOFING, equal error rate, lower is better")
    print("evaluated on:", asr_.get("evaluated_on"))
    print(asr_.get("evaluated_on_note", ""), "\n")
    cond = asr_.get("default_condition", "clean")
    feats = asr_.get("feature_sets", [])
    models = asr_.get("models", [])
    hdr = f"{'feature':10s}" + "".join(f"{m.upper():>10s}" for m in models)
    print(hdr); print("-" * len(hdr))
    for f in feats:
        row = f"{f.upper():10s}"
        for m in models:
            v = (asr_.get("eer", {}).get(f, {}).get(m, {}) or {}).get(cond)
            row += f"{('n/a' if v is None else format(v, '.1%')):>10s}"
        print(row)


In [ ]:
ner = load("ner_results.json")
if ner:
    print("ENTITY RECOGNITION, entity-level F1, exact span match\n")
    for model, srcs in (ner.get("overall") or {}).items():
        for src, v in srcs.items():
            print(f"  {model.upper():8s} on {src:5s} transcripts   "
                  f"P {v.get('p', 0):.3f}  R {v.get('r', 0):.3f}  F1 {v.get('f1', 0):.3f}")

it = load("intent_results.json")
if it:
    print("\nINTENT\n")
    for name, v in (it.get("models") or {}).items():
        print(f"  {name:8s}  AUC {v.get('auc', 0):.3f}  acc {v.get('accuracy', 0):.1%}  F1 {v.get('f1', 0):.3f}")

ttd = load("ttd_results.json")
if ttd:
    print(f"\nTIME TO DETECTION: median {ttd.get('median')}s, "
          f"{ttd.get('detected')}/{ttd.get('total')} scam calls flagged")

cal = load("calibration.json")
if cal:
    print(f"CALIBRATION: Brier {cal.get('brier')}, ECE {cal.get('ece')}")

ood = load("robocall_ood.json")
if ood:
    print()
    print(f"OUT OF DOMAIN: {ood['n_robocalls']} real FTC robocall transcripts")
    print(f"  languages: {ood.get('n_by_language')}")
    for name, v in (ood.get("models") or {}).items():
        print(f"  {name:8s}  recall at threshold {v.get('recall_at_threshold', 0):.1%}  "
              f"({v.get('n_detected')} of {ood['n_robocalls']})  "
              f"median score {v.get('score_median', 0):.3f}")
        for lang, b in (v.get("by_language") or {}).items():
            print(f"            {lang}: {b['recall']:.1%} of {b['n']}")
    print("  " + ood.get("note", ""))


In [ ]:
# Plots for the report.
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(1, 3, figsize=(16, 4.2))

if ab:
    arms = ab.get("arms", []); cs = ab.get("cells", [])
    M = np.array([[ab["detection_rate"].get(a, {}).get(c) or 0 for c in cs] for a in arms])
    im = axes[0].imshow(M, cmap="Blues", vmin=0, vmax=1)
    axes[0].set_xticks(range(len(cs)), [CELL.get(c, c) for c in cs], rotation=25, ha="right")
    axes[0].set_yticks(range(len(arms)), [ARM.get(a, a) for a in arms])
    for i in range(M.shape[0]):
        for j in range(M.shape[1]):
            axes[0].text(j, i, f"{M[i, j]:.0%}", ha="center", va="center",
                         color="white" if M[i, j] > 0.55 else "black", fontsize=9)
    axes[0].set_title("Ablation: correct decisions per cell")

if asr_ and asr_.get("det"):
    for d in asr_["det"][:4]:
        pts = np.array(d["points"])
        axes[1].plot(pts[:, 0], pts[:, 1], label=f"{d['name']} (EER {d['eer']:.1%})")
    axes[1].plot([0, 1], [0, 1], "--", color="grey", lw=1)
    axes[1].set_xlim(0, .6); axes[1].set_ylim(0, .6)
    axes[1].set_xlabel("false alarm rate"); axes[1].set_ylabel("miss rate")
    axes[1].set_title("DET curves"); axes[1].legend(fontsize=8)

if ttd and ttd.get("values"):
    axes[2].hist(ttd["values"], bins=14, color="#3987e5")
    if ttd.get("median"):
        axes[2].axvline(ttd["median"], color="#d03b3b", ls="--",
                        label=f"median {ttd['median']:.1f}s")
        axes[2].legend()
    axes[2].set_xlabel("seconds heard before the alert"); axes[2].set_ylabel("calls")
    axes[2].set_title("Time to detection")

plt.tight_layout()
plt.savefig("data/results/summary.png", dpi=140, bbox_inches="tight")
plt.show()


In [ ]:
# Everything into Drive one more time, so the zip below is a convenience and
# not the only copy. Unchanged artifacts are recognised by file count and total
# size and are not re-uploaded.
colab.save_all()
print()
colab.print_status()


## 6. Take it home

Downloads two zips. Unzip both into your local checkout, then:

```
swarkavach serve
```

and the console shows these numbers with no retraining.


In [ ]:
import json, platform, shutil, subprocess
from datetime import datetime, timezone
from pathlib import Path

# Stamp the provenance so the console can say which numbers came from Colab.
gpu = "cpu"
try:
    import torch
    if torch.cuda.is_available():
        gpu = torch.cuda.get_device_name(0)
except Exception:
    pass

prov = Path("data/results/provenance.json")
entries = []
if prov.exists():
    try:
        entries = json.loads(prov.read_text()).get("entries", [])
    except Exception:
        entries = []
stamp = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M UTC")
for p in sorted(Path("data/results").glob("*.json")):
    if p.name == "provenance.json":
        continue
    entries = [e for e in entries if e.get("artifact") != p.name]
    entries.append({"artifact": p.name, "source": "colab",
                    "generated": stamp, "notes": f"Colab, {gpu}"})
prov.write_text(json.dumps({"entries": entries, "generated": stamp,
                            "host": platform.platform(), "gpu": gpu}, indent=2))

shutil.make_archive("swarkavach_results", "zip", "data/results")
if Path("data/models").exists():
    shutil.make_archive("swarkavach_models", "zip", "data/models")

for f in ("swarkavach_results.zip", "swarkavach_models.zip"):
    if Path(f).exists():
        print(f, "%.1f MB" % (Path(f).stat().st_size / 1e6))

try:
    from google.colab import files
    files.download("swarkavach_results.zip")
    if Path("swarkavach_models.zip").exists():
        files.download("swarkavach_models.zip")
except Exception as exc:
    print("not on Colab, or the download was blocked:", exc)
    print("the zips are in the file browser on the left")


## 7. Push the results straight back to the repo

Optional, and nicer than downloading zips: commit the results and the trained
models to a branch, then pull them on your laptop. Needs the token to have
Contents: read and write.


In [ ]:
PUSH_BACK = True          # set False if you would rather just use the zips
RESULT_BRANCH = "colab-results"

if PUSH_BACK and TOKEN:
    import subprocess
    from datetime import datetime, timezone

    def git(*a, **kw):
        return subprocess.run(["git", *a], capture_output=True, text=True, **kw)

    git("config", "user.email", "colab@swarkavach.local")
    git("config", "user.name", "SwarKavach Colab")

    # models are gitignored by default, so force-add just the small artefacts
    git("checkout", "-B", RESULT_BRANCH)
    git("add", "-f", "data/results")
    for pat in ("data/models/*.joblib", "data/models/*.json", "data/models/*.pt"):
        git("add", "-f", pat)

    stamp = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M UTC")
    c = git("commit", "-m", f"Colab run {stamp}: results and trained models")
    if c.returncode != 0 and "nothing to commit" in (c.stdout + c.stderr):
        print("nothing changed")
    else:
        git("remote", "set-url", "origin", _auth_url())
        r = git("push", "-u", "origin", RESULT_BRANCH, "--force")
        git("remote", "set-url", "origin", CLEAN_URL)
        if r.returncode == 0:
            print(f"pushed to branch {RESULT_BRANCH}")
            print(f"on your laptop:  git fetch origin && git checkout {RESULT_BRANCH}")
        else:
            msg = (r.stderr or "").replace(TOKEN, "***")
            print("push failed:", msg[-600:])
            print("the zips above still work")
elif PUSH_BACK:
    print("no token, so nothing to push. Use the zips above.")


## What to do with it

1. Unzip `swarkavach_results.zip` into `data/results/` in your local checkout.
2. Unzip `swarkavach_models.zip` into `data/models/`.
3. `swarkavach serve`, open <http://127.0.0.1:7860>, go to the Results lab.
   The provenance table will show these rows as coming from Colab.

The other notebooks in this folder go further than this one: `02` trains
RawNet on real ASVspoof data, `03` measures what a Whisper transcript costs
the entity taggers and fine-tunes MuRIL, `05` does consented voice cloning.
This notebook is the one that gets you a complete working system.
